In [1]:
import pandas as pd
import sqlite3

In [2]:
#connect to database connection
connection = sqlite3.connect('ecomm.db')

In [3]:
tables = pd.read_sql_query("SELECT name from sqlite_master WHERE type='table'", connection)
tables

,name
0,olist_customers_dataset
1,olist_geolocation_dataset
2,olist_orders_dataset
3,olist_order_items_dataset
4,olist_order_payments_dataset
5,olist_order_reviews_dataset
6,olist_products_dataset
7,olist_sellers_dataset
8,product_category_name_translation


In [4]:
customers = pd.read_sql("select * from olist_customers_dataset", connection)
customers
geolocation = pd.read_sql("select * from olist_geolocation_dataset", connection)
geolocation
orders = pd.read_sql("select * from olist_orders_dataset", connection)
orders
items = pd.read_sql("select * from olist_order_items_dataset", connection)
items
payments = pd.read_sql("select * from olist_order_payments_dataset", connection)
payments
reviews = pd.read_sql("select * from olist_order_reviews_dataset", connection)
reviews
products = pd.read_sql("select * from olist_products_dataset", connection)
products
sellers = pd.read_sql("select * from olist_sellers_dataset", connection)
sellers

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
...,...,...,...,...
3090,98dddbc4601dd4443ca174359b237166,87111,sarandi,PR
3091,f8201cab383e484733266d1906e2fdfa,88137,palhoca,SC
3092,74871d19219c7d518d0090283e03c137,4650,sao paulo,SP
3093,e603cf3fec55f8697c9059638d6c8eb5,96080,pelotas,RS


In [83]:
#orders.columns
#items.columns
#payments.columns
#products.columns
#sellers.columns
#reviews.columns
customers.columns

Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='object')

In [66]:
#Top payment types by payment value and number of orders
query1 = pd.read_sql("""select p.payment_type, sum(p.payment_value) as total_amount, count(o.order_id) as total_orders 
from olist_order_payments_dataset as p
join olist_orders_dataset as o on p.order_id = o.order_id group by p.payment_type""", connection)
query1

,payment_type,total_amount,total_orders
0,boleto,2869361.27,19784
1,credit_card,12542084.19,76795
2,debit_card,217989.79,1529
3,not_defined,0.00,3
4,voucher,379436.87,5775


In [5]:
# Number of orders in each product category along with average rating
query2 = pd.read_sql("""
    SELECT 
        p.product_category_name, 
        COUNT(o.order_id) AS total_orders, 
        round(avg(r.review_score),2) AS average_rating
    FROM olist_orders_dataset AS o
    JOIN olist_order_reviews_dataset AS r ON r.order_id = o.order_id
    JOIN olist_order_items_dataset AS oi ON oi.order_id = o.order_id
    JOIN olist_products_dataset AS p ON p.product_id = oi.product_id
    WHERE product_category_name != 'None'
    GROUP BY p.product_category_name
    ORDER BY avg(r.review_score) DESC
    limit 10
""", connection)
query2

,product_category_name,total_orders,average_rating
0,cds_dvds_musicais,14,4.64
1,fashion_roupa_infanto_juvenil,8,4.50
2,livros_interesse_geral,549,4.45
3,construcao_ferramentas_ferramentas,99,4.44
4,flores,31,4.42
5,livros_importados,60,4.40
6,livros_tecnicos,266,4.37
7,alimentos_bebidas,279,4.32
8,malas_acessorios,1088,4.32
9,portateis_casa_forno_e_cafe,76,4.30


In [81]:
#States with highest number of orders
query3 = pd.read_sql("""select c.customer_state, count(o.order_id) as total_orders 
from olist_orders_dataset as o
join olist_customers_dataset as c on c.customer_id = o.customer_id 
group by c.customer_state 
order by count(o.order_id) DESC
limit 5""", connection)
query3

,customer_state,total_orders
0,SP,41746
1,RJ,12852
2,MG,11635
3,RS,5466
4,PR,5045


In [73]:
#Freight value according to product category
query4 = pd.read_sql("""select p.product_category_name, sum(i.freight_value) as total_freight 
from olist_products_dataset as p
join olist_order_items_dataset as i on p.product_id = i.product_id group by p.product_category_name 
order by sum(i.freight_value) DESC
limit 5""", connection)
query4

,product_category_name,total_freight
0,cama_mesa_banho,204693.04
1,beleza_saude,182566.73
2,moveis_decoracao,172749.30
3,esporte_lazer,168607.51
4,informatica_acessorios,147318.08


In [82]:
#Total revenue according to each product category
query5 = pd.read_sql("""select p.product_category_name, sum(i.price) as total_revenue 
from olist_products_dataset as p
join olist_order_items_dataset as i on p.product_id = i.product_id group by p.product_category_name 
order by sum(i.price) DESC
limit 10""", connection)
query5

,product_category_name,total_revenue
0,beleza_saude,1258681.34
1,relogios_presentes,1205005.68
2,cama_mesa_banho,1036988.68
3,esporte_lazer,988048.97
4,informatica_acessorios,911954.32
5,moveis_decoracao,729762.49
6,cool_stuff,635290.85
7,utilidades_domesticas,632248.66
8,automotivo,592720.11
9,ferramentas_jardim,485256.46


In [93]:
#Customer Segmentation
query6 = pd.read_sql("""
    WITH customer_category AS (
        SELECT 
            c.customer_unique_id, 
            COUNT(o.order_id) AS order_count,
            CASE 
                WHEN COUNT(o.order_id) = 1 THEN 'New'
                WHEN COUNT(o.order_id) BETWEEN 2 AND 10 THEN 'Returning'
                ELSE 'Loyal'
            END AS customer_segment
        FROM olist_orders_dataset AS o
        JOIN olist_customers_dataset AS c ON o.customer_id = c.customer_id
        GROUP BY c.customer_unique_id
    )
    SELECT 
        customer_segment, 
        COUNT(*) AS number_of_customers
    FROM customer_category
    GROUP BY customer_segment""", connection)
query6

,customer_segment,number_of_customers
0,Loyal,1
1,New,93099
2,Returning,2996
